In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the monthly merged data 
df = pd.read_csv('../data/processed/dengue_weather_monthly.csv')
df['month'] = pd.to_datetime(df['month'])
print(f"Data loaded: {df.shape}")
print(f"Date range: {df['month'].min()} to {df['month'].max()}")
df.head()


Data loaded: (132, 7)
Date range: 2012-01-01 00:00:00 to 2022-12-01 00:00:00


,month,cases,rainfall,temperature,year,month_num,quarter
0,2012-01-01,338,106.1,24.3,2012,1,1
1,2012-02-01,258,83.6,24.8,2012,2,1
2,2012-03-01,231,313.4,24.4,2012,3,1
3,2012-04-01,363,260.6,24.8,2012,4,2
4,2012-05-01,338,292.0,24.8,2012,5,2


### Create Lag Features

In [2]:
# Create lag features for dengue cases
df['cases_lag_1'] = df['cases'].shift(1)  # 1 month ago
df['cases_lag_2'] = df['cases'].shift(2)  # 2 months ago
df['cases_lag_3'] = df['cases'].shift(3)  # 3 months ago
# Show example
example = df[['month', 'cases', 'cases_lag_1', 'cases_lag_2', 'cases_lag_3']].head(6)
example['month'] = example['month'].dt.strftime('%Y-%m')
print(example.to_string(index=False))

  month  cases  cases_lag_1  cases_lag_2  cases_lag_3
2012-01    338          NaN          NaN          NaN
2012-02    258        338.0          NaN          NaN
2012-03    231        258.0        338.0          NaN
2012-04    363        231.0        258.0        338.0
2012-05    338        363.0        231.0        258.0
2012-06    468        338.0        363.0        231.0


### Create Rolling Average

In [4]:
# Create 3-month rolling average
df['cases_rolling_3'] = df['cases'].shift(1).rolling(window=3).mean()
# Show example
example = df[['month', 'cases', 'cases_rolling_3']].head(8)
example['month'] = example['month'].dt.strftime('%Y-%m')
print(example.to_string(index=False))

  month  cases  cases_rolling_3
2012-01    338              NaN
2012-02    258              NaN
2012-03    231              NaN
2012-04    363       275.666667
2012-05    338       284.000000
2012-06    468       310.666667
2012-07    601       389.666667
2012-08    436       469.000000


### Create Weather Lag Features

In [5]:
df['rainfall_lag_1'] = df['rainfall'].shift(1)  # Previous month's rainfall
df['temp_lag_1'] = df['temperature'].shift(1)  # Previous month's temperature
# Show example
example = df[['month', 'rainfall', 'rainfall_lag_1', 'temperature', 'temp_lag_1']].head(6)
example['month'] = example['month'].dt.strftime('%Y-%m')
print(example.to_string(index=False))

  month  rainfall  rainfall_lag_1  temperature  temp_lag_1
2012-01     106.1             NaN         24.3         NaN
2012-02      83.6           106.1         24.8        24.3
2012-03     313.4            83.6         24.4        24.8
2012-04     260.6           313.4         24.8        24.4
2012-05     292.0           260.6         24.8        24.8
2012-06      53.0           292.0         26.3        24.8


### Create Interaction Feature
#### Testing if combined weather conditions matter

In [7]:
# Create temperature × rainfall interaction (used lagged weather)
df['temp_rain_product'] = df['temp_lag_1'] * df['rainfall_lag_1']
# Show statistics
print("\nInteraction feature statistics:")
print(df['temp_rain_product'].describe())


Interaction feature statistics:
count      131.000000
mean      4124.910076
std       2462.386573
min          4.900000
25%       2370.320000
50%       3723.200000
75%       5463.420000
max      16488.640000
Name: temp_rain_product, dtype: float64


### Handling Missing Values

In [8]:
print("Missing values by column:")
print(df.isnull().sum())

# Count rows before dropping
rows_before = len(df)

# Drop rows with NaN values (from shifting and rolling)
df_clean = df.dropna()

# Count rows after dropping
rows_after = len(df_clean)
rows_lost = rows_before - rows_after

print(f"  Data Summary:")
print(f"  Rows before: {rows_before}")
print(f"  Rows after: {rows_after}")
print(f"  Rows removed: {rows_lost}")
print(f"  Percentage kept: {rows_after/rows_before*100:.1f}%")

print(f"New date range: {df_clean['month'].min().date()} to {df_clean['month'].max().date()}")

Missing values by column:
month                0
cases                0
rainfall             0
temperature          0
year                 0
month_num            0
quarter              0
cases_lag_1          1
cases_lag_2          2
cases_lag_3          3
cases_rolling_3      3
rainfall_lag_1       1
temp_lag_1           1
temp_rain_product    1
dtype: int64
  Data Summary:
  Rows before: 132
  Rows after: 129
  Rows removed: 3
  Percentage kept: 97.7%
New date range: 2012-04-01 to 2022-12-01


In [9]:
# Save the feature-engineered dataset
df_clean.to_csv('../data/processed/dengue_features_final.csv', index=False)

# Feature Engineering 

## Features Created:

### 1. **Lag Features (3)**
- `cases_lag_1` - Cases from 1 month ago
- `cases_lag_2` - Cases from 2 months ago
- `cases_lag_3` - Cases from 3 months ago

**Rationale:** Dengue outbreaks have momentum. High cases last month → likely high cases this month.



### 2. **Rolling Average (1)**
- `cases_rolling_3` - Average of last 3 months

**Rationale:** Smooths noise, captures underlying trends.



### 3. **Weather Lag Features (2)**
- `rainfall_lag_1` - Rainfall from previous month
- `temp_lag_1` - Temperature from previous month

**Rationale:** EDA showed weak correlation for current month weather (-0.058 for rain). Time lag hypothesis: rain → mosquito breeding → dengue appears 1-2 months later.



### 4. **Interaction Feature (1)**
- `temp_rain_product` - temperature × rainfall (Lagged)

**Rationale:** Test if combined warm + rainy conditions matter more than individual factors.



## Dataset Transformation:

**Before:**
- 7 columns (month, cases, rainfall, temperature, year, month_num, quarter)
- ~150 rows

**After:**
- 14 columns (7 original + 7 engineered)
- ~129 rows (lost 3 from NaN handling)
- Date range: 2012-04-01 to 2022-12-01

